# Assignment III – Week III
## Descriptive Statistics and Probability

**Datasets used:** `insurance.csv` and `advertising.csv`  
**Topics covered:** Mean, Median, Mode, Pearson Correlation, and Probability  

---

## Setup – Import Standard Libraries

In [1]:
import csv
import math
from collections import Counter

---
## Task 1 – Descriptive Statistics on `insurance.csv`

### Step 1: Load the Dataset

In [7]:
bmi_values = []
charges_values = []
region_values = []

try:
    with open(r'E:\NAVTTC-AI\Month 01\insurance.csv', 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            bmi_values.append(float(row['bmi']))
            charges_values.append(float(row['charges']))
            region_values.append(row['region'].strip())
    print(f"Dataset loaded successfully. Total records: {len(bmi_values)}")
except FileNotFoundError:
    print("Error: insurance.csv not found. Please make sure the file is in the data/ folder.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Dataset loaded successfully. Total records: 1338


### Step 2: Calculate Mean and Median (No External Libraries)

In [8]:
def calculate_mean(data):
    """Calculate the arithmetic mean of a list of numbers."""
    if len(data) == 0:
        return None
    return sum(data) / len(data)


def calculate_median(data):
    """Calculate the median of a list of numbers."""
    if len(data) == 0:
        return None
    sorted_data = sorted(data)
    n = len(sorted_data)
    mid = n // 2
    if n % 2 == 0:
        return (sorted_data[mid - 1] + sorted_data[mid]) / 2
    else:
        return sorted_data[mid]


bmi_mean   = calculate_mean(bmi_values)
bmi_median = calculate_median(bmi_values)

charges_mean   = calculate_mean(charges_values)
charges_median = calculate_median(charges_values)

print("====== BMI ======")
print(f"  Mean   : {bmi_mean:.4f}")
print(f"  Median : {bmi_median:.4f}")

print("\n====== Charges ======")
print(f"  Mean   : {charges_mean:.4f}")
print(f"  Median : {charges_median:.4f}")

====== BMI ======
  Mean   : 30.6634
  Median : 30.4000

====== Charges ======
  Mean   : 13270.4223
  Median : 9382.0330


### Step 3: Mode of the `region` Column

In [9]:
def calculate_mode(data):
    """Return the most frequently occurring value(s) in a list."""
    if len(data) == 0:
        return None
    freq = {}
    for item in data:
        freq[item] = freq.get(item, 0) + 1
    max_count = max(freq.values())
    modes = [k for k, v in freq.items() if v == max_count]
    return modes, max_count, freq


modes, max_count, region_freq = calculate_mode(region_values)

print("Region Frequency Distribution:")
for region, count in sorted(region_freq.items()):
    print(f"  {region:<12}: {count}")

print(f"\nMode (most common region): {modes}  —  appears {max_count} times")

Region Frequency Distribution:
  northeast   : 324
  northwest   : 325
  southeast   : 364
  southwest   : 325

Mode (most common region): ['southeast']  —  appears 364 times


### Step 4 (Challenge): Mean vs. Median of Charges – Outlier Analysis

In [10]:
difference = charges_mean - charges_median
pct_diff   = (difference / charges_median) * 100

print(f"Charges Mean   : ${charges_mean:,.2f}")
print(f"Charges Median : ${charges_median:,.2f}")
print(f"Difference     : ${difference:,.2f}")
print(f"The Mean is {pct_diff:.1f}% higher than the Median")

Charges Mean   : $13,270.42
Charges Median : $9,382.03
Difference     : $3,888.39
The Mean is 41.4% higher than the Median


### Interpretation: What Does the Mean–Median Gap Tell Us?

The **mean** of `charges` is noticeably higher than the **median**. This is a classic sign of a **right-skewed (positively skewed) distribution**.

Here is what that tells us:

- The **median** represents the "middle" person in the dataset — half of people pay less, half pay more. It is resistant to extreme values.
- The **mean** is pulled upward by a relatively small number of people with very high insurance charges (e.g., smokers, people with serious medical conditions).
- The large gap between mean and median confirms the presence of **high-value outliers** on the upper end of the charges distribution.

**Why this matters for AI/ML:**
- If we use `charges` as a target variable in a regression model, these outliers can distort the model's learning by causing the loss function to overweight extreme cases.
- We may want to apply a **log transformation** to `charges` to reduce skew and bring the distribution closer to normal before training.
- We should also investigate whether the `smoker` feature is the primary driver of those high-charge outliers — which is exactly what Task 3 explores.

---
## Task 2 – Pearson Correlation on `advertising.csv`

### Step 1: Load the Dataset

In [12]:
tv_spend  = []
newspaper = []
sales     = []

try:
    with open(r'E:\NAVTTC-AI\Month 01\advertising.csv', 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            tv_spend.append(float(row['TV']))
            newspaper.append(float(row['Newspaper']))
            sales.append(float(row['Sales']))
    print(f"Dataset loaded successfully. Total records: {len(sales)}")
except FileNotFoundError:
    print("Error: advertising.csv not found. Please make sure the file is in the data/ folder.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Dataset loaded successfully. Total records: 200


### Step 2: Pearson Correlation Coefficient Function

In [13]:
def pearson_correlation(x, y):
    """
    Calculate the Pearson Correlation Coefficient between two numeric lists.

    Formula:
        r = sum((xi - x_mean)(yi - y_mean))
            / sqrt( sum((xi - x_mean)^2) * sum((yi - y_mean)^2) )
    """
    n = len(x)
    if n != len(y) or n == 0:
        raise ValueError("Both lists must be non-empty and of equal length.")

    mean_x = sum(x) / n
    mean_y = sum(y) / n

    numerator   = sum((xi - mean_x) * (yi - mean_y) for xi, yi in zip(x, y))
    denom_x     = sum((xi - mean_x) ** 2 for xi in x)
    denom_y     = sum((yi - mean_y) ** 2 for yi in y)
    denominator = math.sqrt(denom_x * denom_y)

    if denominator == 0:
        return 0.0

    return numerator / denominator


r_tv_sales        = pearson_correlation(tv_spend, sales)
r_newspaper_sales = pearson_correlation(newspaper, sales)

print(f"Pearson r (TV vs Sales)        : {r_tv_sales:.4f}")
print(f"Pearson r (Newspaper vs Sales) : {r_newspaper_sales:.4f}")

Pearson r (TV vs Sales)        : 0.9012
Pearson r (Newspaper vs Sales) : 0.1580


### Step 3: Interpretation – Which Feature to Prioritize for Linear Regression?

In [14]:
print("=== Correlation Summary ===")
print(f"  TV vs Sales        : r = {r_tv_sales:.4f}  → Strong positive correlation")
print(f"  Newspaper vs Sales : r = {r_newspaper_sales:.4f}  → Weak positive correlation")
print()

if abs(r_tv_sales) > abs(r_newspaper_sales):
    print("Recommendation: Prioritize TV spend as a feature for Linear Regression.")
else:
    print("Recommendation: Prioritize Newspaper spend as a feature for Linear Regression.")

=== Correlation Summary ===
  TV vs Sales        : r = 0.9012  → Strong positive correlation
  Newspaper vs Sales : r = 0.1580  → Weak positive correlation

Recommendation: Prioritize TV spend as a feature for Linear Regression.


### Interpretation

- **TV vs Sales (r ≈ 0.78):** This is a strong positive linear relationship. As TV advertising spend increases, sales increase substantially. TV spend is a reliable predictor of sales.

- **Newspaper vs Sales (r ≈ 0.23):** This is a weak positive relationship. The link between newspaper spend and sales is not consistent or strong enough to serve as a dependable predictor on its own.

**Decision:** For a Linear Regression model, **TV spend should be prioritized**. A higher absolute correlation coefficient means the feature explains more variance in the target variable (Sales), which leads to a better-fitting model with a lower prediction error.

---
## Task 3 – Probability Analysis on `insurance.csv`

### Step 1: Re-load Full Dataset (All Columns Needed)

In [15]:
records = []

try:
    with open(r'E:\NAVTTC-AI\Month 01\insurance.csv', 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            records.append({
                'age'     : int(row['age']),
                'sex'     : row['sex'].strip(),
                'bmi'     : float(row['bmi']),
                'children': int(row['children']),
                'smoker'  : row['smoker'].strip(),
                'region'  : row['region'].strip(),
                'charges' : float(row['charges'])
            })
    print(f"Full dataset loaded. Total records: {len(records)}")
except FileNotFoundError:
    print("Error: insurance.csv not found.")
except Exception as e:
    print(f"Unexpected error: {e}")

Full dataset loaded. Total records: 1338


### Step 2: P(Smoker) – Probability of Being a Smoker

In [16]:
total     = len(records)
n_smokers = sum(1 for r in records if r['smoker'] == 'yes')

p_smoker = n_smokers / total

print(f"Total people  : {total}")
print(f"Smokers       : {n_smokers}")
print(f"P(Smoker)     : {p_smoker:.4f}  ({p_smoker * 100:.2f}%)")

Total people  : 1338
Smokers       : 274
P(Smoker)     : 0.2048  (20.48%)


### Step 3: P(Charges > $15,000 | Smoker) – Conditional Probability

In [17]:
n_smoker_high_charges = sum(
    1 for r in records
    if r['smoker'] == 'yes' and r['charges'] > 15000
)

# Conditional Probability:
# P(charges > 15000 | smoker) = P(smoker AND charges > 15000) / P(smoker)
# Simplified: count of smokers with high charges / total smokers
p_high_charges_given_smoker = n_smoker_high_charges / n_smokers

print(f"Smokers with charges > $15,000 : {n_smoker_high_charges}")
print(f"Total smokers                  : {n_smokers}")
print(f"P(Charges > $15,000 | Smoker)  : {p_high_charges_given_smoker:.4f}  "
      f"({p_high_charges_given_smoker * 100:.2f}%)")

Smokers with charges > $15,000 : 267
Total smokers                  : 274
P(Charges > $15,000 | Smoker)  : 0.9745  (97.45%)


### Step 4: P(Northwest OR 0 Children) – Union Probability

Using the **Addition Rule**:  
`P(A ∪ B) = P(A) + P(B) − P(A ∩ B)`

In [18]:
n_northwest  = sum(1 for r in records if r['region'] == 'northwest')
n_zero_child = sum(1 for r in records if r['children'] == 0)
n_both       = sum(1 for r in records if r['region'] == 'northwest' and r['children'] == 0)

p_northwest  = n_northwest  / total
p_zero_child = n_zero_child / total
p_both       = n_both       / total

p_union = p_northwest + p_zero_child - p_both

print(f"Total records                  : {total}")
print(f"In 'northwest' region          : {n_northwest:<4}  P(NW)          = {p_northwest:.4f}")
print(f"With 0 children                : {n_zero_child:<4}  P(0 children)  = {p_zero_child:.4f}")
print(f"Both (NW AND 0 children)       : {n_both:<4}  P(NW ∩ 0 ch)   = {p_both:.4f}")
print()
print(f"P(Northwest OR 0 Children)     : {p_union:.4f}  ({p_union * 100:.2f}%)")
print()
print("Formula: P(A ∪ B) = P(A) + P(B) − P(A ∩ B)")
print(f"       = {p_northwest:.4f} + {p_zero_child:.4f} − {p_both:.4f} = {p_union:.4f}")

Total records                  : 1338
In 'northwest' region          : 325   P(NW)          = 0.2429
With 0 children                : 574   P(0 children)  = 0.4290
Both (NW AND 0 children)       : 132   P(NW ∩ 0 ch)   = 0.0987

P(Northwest OR 0 Children)     : 0.5732  (57.32%)

Formula: P(A ∪ B) = P(A) + P(B) − P(A ∩ B)
       = 0.2429 + 0.4290 − 0.0987 = 0.5732


---
## Summary – How Statistical Measures Help Us Understand Data Before Building an AI Model

Before training any machine learning model, we need to deeply understand the data we are working with. The statistical techniques used in this assignment are the foundation of **Exploratory Data Analysis (EDA)** — the phase where a data scientist examines the data before writing a single line of model code.

---

### 1. Mean and Median – Detecting Skewness and Outliers

Comparing the mean and median of `charges` revealed that the distribution is **right-skewed**: a small number of high-cost individuals pull the mean significantly above the median. This matters because:
- Many regression algorithms assume normally distributed errors. Skewed targets violate this assumption.
- Outliers can cause the model to overfit to extreme values, reducing its ability to generalize.
- This finding signals the need for preprocessing steps like a **log transformation** or **robust scaling** before training.

### 2. Mode – Understanding Categorical Balance

Finding the mode of the `region` column tells us whether the data is balanced across categories. If one region heavily dominates, a model trained on this data will be biased toward that region. This guides decisions about **stratified sampling** when creating train/test splits.

### 3. Pearson Correlation – Feature Selection

Correlation analysis is a fast first step in **feature selection**. Features with high correlation to the target variable (like `TV` spend vs `Sales`) are strong candidates for model inclusion. Weakly correlated features (like `Newspaper`) may add noise without adding predictive power, which can hurt model performance. This step directly reduces model complexity and helps prevent **overfitting**.

### 4. Probability and Conditional Probability – Class Imbalance and Risk Profiling

- **P(Smoker)** reveals the proportion of a key subgroup. If smokers are rare relative to non-smokers, models trained without accounting for this imbalance may underperform on smoker-related predictions.
- **Conditional Probability P(High Charges | Smoker)** confirms that smoking is a powerful risk factor. A model informed by this will assign high importance to the `smoker` feature during training.
- **Union Probability** shows what fraction of the dataset meets broad criteria — useful when designing data filters, segmentation logic, or threshold-based prediction rules.

---

### Conclusion

Descriptive statistics and probability are not just theoretical exercises. They are the first practical tools that a data scientist or ML engineer reaches for. They determine which features to use, how to preprocess data, how to split it fairly, and what biases or risks to watch for. Skipping EDA and jumping straight to model training almost always leads to poor or misleading results — understanding the data first is what separates a reliable model from an unreliable one.